Job Search AI Agent
-------------------
A simple agentic AI system that:
1. Reads your resume and converts it to structured JSON
2. Searches Indeed for matching jobs
3. Creates tailored cover letters and resumes for each job
4. Tracks which jobs it has already processed


In [ ]:
# Imports
import os
import json
import time
import gradio as gr
from openai import OpenAI
import requests
from bs4 import BeautifulSoup

from datetime import datetime

from dotenv import load_dotenv
load_dotenv(override=True)

import nest_asyncio
nest_asyncio.apply()  # Allows Gradio to work inside Jupyter's event loop

In [ ]:
# Configuration

# Location to save tailored resumes and cover letters
OUTPUT_FOLDER = "job_applications"

# File that tracks which jobs we've already processed
PROCESSED_JOBS_FILE = "processed_jobs.json"

MODEL = "anthropic/claude-3.5-sonnet"

In [ ]:
# Creating Client instance via a function
def get_client() -> OpenAI:
    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ.get("OPENROUTER_API_KEY", ""),
    )

In [54]:
import concurrent.futures

def read_resume(file_path: str) -> str:
    """Read text from a PDF or DOCX resume file — runs in a thread to avoid event loop conflicts in Jupyter."""
    
    def _read():
        if file_path.endswith(".pdf"):
            from pypdf import PdfReader
            reader = PdfReader(file_path)
            return "".join(page.extract_text() or "" for page in reader.pages)
        elif file_path.endswith(".docx"):
            from docx import Document
            doc = Document(file_path)
            return "\n".join([para.text for para in doc.paragraphs])
        else:
            with open(file_path, "r", encoding="utf-8") as f:
                return f.read()
    
    with concurrent.futures.ThreadPoolExecutor() as pool:
        future = pool.submit(_read)
        return future.result()

In [55]:
# Agent 1: Resume parser with Structured JSON output

def agent_parse_resume(resume_text: str) -> dict:
    """
    Agent 1: Takes raw resume text and returns a structured JSON
    with skills, experience, education, etc.
    """
    print("🤖 Agent 1: Parsing resume into structured JSON...")
    
    system_prompt = """ 
                    You are an expert HR analyst and resume specialist with 15+ years of experience 
                    parsing resumes across all industries. You extract structured information with 
                    precision and always return clean, valid JSON — no extra commentary, no markdown.
                    """
    user_prompt = f"""Extract the following information from this resume and return ONLY valid JSON (no markdown, no explanation):
                      {{
                        "name": "candidate full name",
                        "email": "email address",
                        "phone": "phone number",
                        "location": "city, state/country",
                        "summary": "brief professional summary",
                        "skills": ["skill1", "skill2", "..."],
                        "experience": [
                          {{
                            "title": "job title",
                            "company": "company name",
                            "duration": "start - end",
                            "description": "what they did"
                          }}
                        ],
                        "education": [
                          {{
                            "degree": "degree name",
                            "school": "school name",
                            "year": "graduation year"
                          }}
                        ],
                        "job_titles_seeking": ["list of job titles that match this resume"],
                        "industry": "primary industry"
                      }}

                      Resume text:
                      {resume_text}
                  """

    Message = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
              ]

    response = get_client().chat.completions.create(
        model=MODEL,
        max_tokens=2000,
        messages= Message
      )

    raw = response.choices[0].message.content.strip()
    # Clean up if model wrapped in markdown
    raw = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(raw)




In [56]:
# Agent 2: Searches Indeed for Jobs

def scrape_indeed_jobs(job_title: str, location: str = "United States", num_jobs: int = 10) -> list[dict]:
    """
    Scrape Indeed job listings for a given job title.
    Returns a list of job dicts with title, company, location, link, and description snippet.
    
    Note: Indeed may block automated scraping. This uses a simple requests approach.
    If blocked, consider using their official API or a job search API service.
    """
    print(f"🔍 Searching Indeed for: {job_title}")

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    search_query = job_title.replace(" ", "+")
    url = f"https://www.indeed.com/jobs?q={search_query}&l={location.replace(' ', '+')}"

    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        jobs = []
        job_cards = soup.find_all("div", class_="job_seen_beacon")

        for card in job_cards[:num_jobs]:
            try:
                title_tag = card.find("h2", class_="jobTitle")
                title = title_tag.get_text(strip=True) if title_tag else "N/A"

                company_tag = card.find("span", {"data-testid": "company-name"})
                company = company_tag.get_text(strip=True) if company_tag else "N/A"

                location_tag = card.find("div", {"data-testid": "text-location"})
                job_location = location_tag.get_text(strip=True) if location_tag else "N/A"

                link_tag = title_tag.find("a") if title_tag else None
                job_id = link_tag["data-jk"] if link_tag and link_tag.get("data-jk") else ""
                job_link = f"https://www.indeed.com/viewjob?jk={job_id}" if job_id else url

                snippet_tag = card.find("div", {"data-testid": "jobsnippet_footer"})
                if not snippet_tag:
                    snippet_tag = card.find("div", class_="job-snippet")
                snippet = snippet_tag.get_text(strip=True) if snippet_tag else "No description available"

                jobs.append({
                    "title": title,
                    "company": company,
                    "location": job_location,
                    "link": job_link,
                    "description": snippet,
                    "job_id": job_id or f"{company}_{title}".replace(" ", "_")
                })
            except Exception:
                continue

        return jobs

    except Exception as e:
        print(f"⚠️  Could not scrape Indeed: {e}")
        # Return mock data so the rest of the flow still works during development
        # in situtations where Indeed doesn't allow access
        return [
            {
                "title": job_title,
                "company": "Example Corp",
                "location": location,
                "link": "https://www.indeed.com",
                "description": f"Exciting {job_title} role with growth opportunities.",
                "job_id": f"mock_{i}"
            }
            for i in range(5)
        ]


def agent_find_jobs(resume_json: dict) -> list[dict]:
    """
    Agent 2: Uses the structured resume to search for matching jobs on Indeed.
    Returns up to 10 job postings.
    """
    print("🤖 Agent 2: Finding matching jobs...")

    all_jobs = []
    titles_to_search = resume_json.get("job_titles_seeking", ["Software Engineer"])[:2]

    for title in titles_to_search:
        jobs = scrape_indeed_jobs(title, num_jobs=5)
        all_jobs.extend(jobs)
        time.sleep(1)

    return all_jobs[:10]

In [57]:
# Agent 3: Creates Tailored Resume & Cover Letter 

def agent_create_application_materials(
    resume_json: dict,
    original_resume_text: str,
    job: dict
) -> tuple[str, str]:
    """
    Agent 3: Creates a tailored cover letter and resume for a specific job.
    Returns (cover_letter_text, tailored_resume_text).
    """
    print(f"✍️  Agent 3: Creating materials for {job['title']} at {job['company']}...")

    system_prompt = """ 
                    You are a senior career coach and professional resume writer with a track record 
                    of helping candidates land roles at top companies. You write tailored, compelling 
                    cover letters and resumes that highlight the strongest match between a candidate's 
                    background and the specific job. Your writing is confident, concise, and human but 
                    never generic or overly formal. You always follow the exact output format requested.
                    """

    user_prompt = f"""You are an expert career coach and resume writer.

                    Candidate profile (JSON):
                    {json.dumps(resume_json, indent=2)}

                    Original resume:
                    {original_resume_text[:3000]}

                    Job posting:
                    Title: {job['title']}
                    Company: {job['company']}
                    Location: {job['location']}
                    Description: {job['description']}

                    Please create TWO things and return them in this EXACT format:

                    ===COVER_LETTER===
                    [Write a professional, personalized cover letter of 3-4 paragraphs. 
                    Highlight skills that match the job. Keep it warm and human.]

                    ===TAILORED_RESUME===
                    [Rewrite the resume to emphasize skills and experience most relevant 
                    to this specific job. Keep all facts accurate - just reorder and 
                    reword to match the job requirements better.]
                    """

    response = get_client().chat.completions.create(
        model=MODEL,
        max_tokens=3000,
        messages=[{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
    )

    full_text = response.choices[0].message.content

    # Split into cover letter and resume
    parts = full_text.split("===TAILORED_RESUME===")
    cover_letter = parts[0].replace("===COVER_LETTER===", "").strip()
    tailored_resume = parts[1].strip() if len(parts) > 1 else "Resume not generated."

    return cover_letter, tailored_resume

In [58]:
# Processed Jobs Tracker

def load_processed_jobs() -> dict:
    """Load the list of jobs we've already processed."""
    if os.path.exists(PROCESSED_JOBS_FILE):
        with open(PROCESSED_JOBS_FILE, "r") as f:
            return json.load(f)
    return {}

In [59]:
def save_processed_job(job: dict):
    """Mark a job as processed so we skip it next time."""
    processed = load_processed_jobs()
    processed[job["job_id"]] = {
        "title": job["title"],
        "company": job["company"],
        "processed_at": datetime.now().isoformat()
    }
    with open(PROCESSED_JOBS_FILE, "w") as f:
        json.dump(processed, f, indent=2)


In [60]:
def reset_processed_jobs():
    """Clear the processed jobs tracker (called when user clicks Reset)."""
    if os.path.exists(PROCESSED_JOBS_FILE):
        os.remove(PROCESSED_JOBS_FILE)
    print("🔄 Reset: All processed jobs cleared.")

In [61]:
# Save to Disk

def save_application_materials(job: dict, cover_letter: str, tailored_resume: str):
    """Save cover letter and resume to the output folder."""
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    # Create a safe folder name from company + title
    safe_name = f"{job['company']}_{job['title']}".replace(" ", "_").replace("/", "-")[:50]
    job_folder = os.path.join(OUTPUT_FOLDER, safe_name)
    os.makedirs(job_folder, exist_ok=True)

    with open(os.path.join(job_folder, "cover_letter.txt"), "w") as f:
        f.write(cover_letter)

    with open(os.path.join(job_folder, "tailored_resume.txt"), "w") as f:
        f.write(tailored_resume)

    print(f"💾 Saved materials to: {job_folder}")

In [ ]:
# Main Orchestrator
def run_job_search(resume_file, reset: bool = False):
    """
    Generator orchestrator — yields the growing output string after every step
    so Gradio streams it live to the browser.
    """
    if resume_file is None:
        yield "❌ Please upload a resume file first."
        return

    if reset:
        reset_processed_jobs()

    processed = load_processed_jobs()
    output = ""

    def stream(line):
        """Append a line to output and return the full text so far."""
        nonlocal output
        output += line + "\n"
        return output

    yield stream("🚀 Starting Job Search Agent...\n")

    # Step 1: Parse resume 
    yield stream("📄 **Step 1:** Reading and parsing your resume...")
    resume_text = read_resume(resume_file)
    yield stream("⏳ Resume loaded — sending to AI for parsing...")
    resume_json = agent_parse_resume(resume_text)
    yield stream("\n" + f"✅ Resume parsed! Hello, **{resume_json.get('name', 'there')}**!")
    yield stream("\n" + f"🧠 Skills found: {', '.join(resume_json.get('skills', [])[:5])}\n")

    # Step 2: Find jobs
    yield stream("🔍 **Step 2:** Searching for matching jobs on Indeed...")
    all_jobs = agent_find_jobs(resume_json)
    new_jobs = [j for j in all_jobs if j["job_id"] not in processed]
    yield stream(f"📋 Found **{len(all_jobs)}** jobs — **{len(new_jobs)}** are new.\n")

    if not new_jobs:
        yield stream("⚠️ All found jobs have already been processed!")
        yield stream("👉 Check **Reset Search** and try again.")
        return

    # Step 3: Create materials for top 5 new jobs
    yield stream("✍️ **Step 3:** Creating tailored applications for top 5 new jobs...\n")
    yield stream("---")
    yield stream("## 📋 Top 5 Job Applications\n")

    top_jobs = new_jobs[:5]
    for i, job in enumerate(top_jobs, 1):
        yield stream(f"### {i}. {job['title']} at {job['company']}")
        yield stream(f"📍 {job['location']}")
        yield stream(f"🔗 [Apply here]({job['link']})")
        yield stream(f"⏳ Generating tailored cover letter and resume...")

        cover_letter, tailored_resume = agent_create_application_materials(
            resume_json, resume_text, job
        )
        save_application_materials(job, cover_letter, tailored_resume)
        save_processed_job(job)

        yield stream(f"✅ Materials saved to `./{OUTPUT_FOLDER}/`\n")

    yield stream("---")
    yield stream(f"🎉 **All done!** Check the `{OUTPUT_FOLDER}/` folder for your tailored documents.")


In [ ]:
# Gradio UI

def gradio_run(resume_file, reset_search):
    """Wrapper for Gradio — passes the generator through so Gradio streams it."""
    if resume_file is None:
        yield "❌ Please upload your resume (PDF, DOCX, or TXT)."
        return
    file_path = resume_file.name if hasattr(resume_file, "name") else resume_file
    yield from run_job_search(file_path, reset=reset_search)


def build_ui():
    with gr.Blocks(title="🤖 Job Search Assistant", theme=gr.themes.Soft()) as demo:

        gr.Markdown("""
        # 🤖 Job Search Assistant
        """)
       
        with gr.Row():
            with gr.Column(scale=1):
                resume_input = gr.File(
                    label="📎 Upload Your Resume",
                    file_types=[".pdf", ".docx", ".txt"],
                    type="filepath"
                )
                reset_checkbox = gr.Checkbox(
                    label="🔄 Reset Search (reprocess already-seen jobs)",
                    value=False
                )
                run_button = gr.Button("🚀 Start Job Search", variant="primary", size="lg")

            with gr.Column(scale=2):
                output_box = gr.Markdown(
                    label="📊 Agent Progress & Results",
                    value="Results will appear here after you click 'Start Job Search'..."
                )

        run_button.click(
            fn=gradio_run,
            inputs=[resume_input, reset_checkbox],
            outputs=output_box,
            show_progress=True
        )

        gr.Markdown("""
        ---
        > **Note:** Tailored resumes and cover letters are saved in the `job_applications/` folder.  
        > Each job gets its own subfolder. The agent remembers which jobs it has already processed  
        > so it won't duplicate work — unless you check 'Reset Search'.
        """)

    return demo

In [ ]:
print("Starting AI Job Search Agent...")
print(f"Applications will be saved to: ./{OUTPUT_FOLDER}/")
app = build_ui()
app.launch(share=False, inbrowser=True)